<a href="https://colab.research.google.com/github/psiperez/budas_trade/blob/main/json_to_odt_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Script para ler arquivos JSON e exportar para ODT no Google Colab
# Execute este script em uma célula do Google Colab

import json
import os
from google.colab import files
from datetime import datetime
import re
import subprocess
import sys

# Instalação da biblioteca odfpy ANTES da importação
def install_odfpy():
    """
    Instala a biblioteca odfpy se não estiver disponível
    """
    try:
        import odf
        print("✅ Biblioteca odfpy já está instalada!")
        return True
    except ImportError:
        print("📦 Instalando biblioteca odfpy...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "odfpy"])
            print("✅ Biblioteca odfpy instalada com sucesso!")
            return True
        except Exception as e:
            print(f"❌ Erro ao instalar odfpy: {e}")
            return False

# Instala e importa as bibliotecas necessárias
ODF_AVAILABLE = install_odfpy()

if not ODF_AVAILABLE:
    print("❌ Não foi possível instalar a biblioteca odfpy. O script será executado em modo limitado.")
    # Cria classes dummy para evitar erros
    class DummyClass:
        def __getattr__(self, name):
            return lambda *args, **kwargs: None

    class DummyStyles:
        def getByName(self, name):
            return None

    class DummyText:
        def addElement(self, element):
            pass

    class DummyDoc:
        def __init__(self):
            self.styles = DummyStyles()
            self.text = DummyText()

    OpenDocumentText = DummyDoc
    Style = DummyClass
    TextProperties = DummyClass
    P = DummyClass
    Span = DummyClass
    H = DummyClass
    Table = DummyClass
    TableRow = DummyClass
    TableCell = DummyClass
else:
    # Importa as bibliotecas após a instalação
    from odf.opendocument import OpenDocumentText
    from odf.style import Style, TextProperties, ParagraphProperties
    from odf.text import P, Span, H
    from odf import table

class JSONtoODTConverter:
    """
    Classe para converter arquivos JSON para ODT com formatação
    """

    def __init__(self):
        self.doc = None
        self.styles = {}
        self.toc_entries = []
        self.odf_available = ODF_AVAILABLE

    def get_style_by_name(self, style_name):
        """
        Método seguro para obter um estilo pelo nome
        """
        if not self.odf_available or not self.doc:
            return None

        try:
            if hasattr(self.doc, 'styles') and hasattr(self.doc.styles, 'getByName'):
                return self.doc.styles.getByName(style_name)
            return None
        except Exception:
            return None

    def create_document(self):
        """Cria um novo documento ODT com estilos personalizados"""
        if not self.odf_available:
            print("⚠️ Biblioteca odfpy não disponível. Criando documento em formato alternativo.")
            return None

        self.doc = OpenDocumentText()

        # Estilo para títulos
        title_style = Style(name="TitleStyle", family="paragraph")
        title_style.addElement(TextProperties(attributes={'fontsize': '20pt', 'fontweight': 'bold'}))
        self.doc.styles.addElement(title_style)

        # Estilo para cabeçalhos de seção
        heading_style = Style(name="HeadingStyle", family="paragraph")
        heading_style.addElement(TextProperties(attributes={'fontsize': '14pt', 'fontweight': 'bold'}))
        self.doc.styles.addElement(heading_style)

        # Estilo para parágrafos normais
        normal_style = Style(name="NormalStyle", family="paragraph")
        normal_style.addElement(TextProperties(attributes={'fontsize': '11pt'}))
        self.doc.styles.addElement(normal_style)

        # Estilo para chaves JSON
        key_style = Style(name="KeyStyle", family="text")
        key_style.addElement(TextProperties(attributes={'fontweight': 'bold', 'color': '#2c3e50'}))
        self.doc.styles.addElement(key_style)

        # Estilo para strings
        string_style = Style(name="StringStyle", family="text")
        string_style.addElement(TextProperties(attributes={'color': '#27ae60', 'fontstyle': 'italic'}))
        self.doc.styles.addElement(string_style)

        # Estilo para números
        number_style = Style(name="NumberStyle", family="text")
        number_style.addElement(TextProperties(attributes={'color': '#2980b9', 'fontweight': 'bold'}))
        self.doc.styles.addElement(number_style)

        # Estilo para booleanos
        boolean_style = Style(name="BooleanStyle", family="text")
        boolean_style.addElement(TextProperties(attributes={'color': '#e74c3c', 'fontweight': 'bold'}))
        self.doc.styles.addElement(boolean_style)

        # Estilo para valores nulos
        null_style = Style(name="NullStyle", family="text")
        null_style.addElement(TextProperties(attributes={'color': '#95a5a6', 'fontstyle': 'italic'}))
        self.doc.styles.addElement(null_style)

        return self.doc

    def add_paragraph(self, text, style_name=None):
        """
        Adiciona um parágrafo ao documento
        CORREÇÃO: Sempre usa P como elemento pai, nunca Span diretamente no text
        """
        if not self.odf_available or not self.doc:
            return None

        p = P()

        # Aplica estilo ao parágrafo se especificado
        if style_name:
            style = self.get_style_by_name(style_name)
            if style:
                p.setAttribute('stylename', style)

        # Adiciona o texto ao parágrafo
        p.addText(text)

        # Adiciona o parágrafo ao documento
        self.doc.text.addElement(p)
        return p

    def add_heading(self, text, level=1):
        """Adiciona um cabeçalho ao documento"""
        if not self.odf_available or not self.doc:
            return None

        h = H(outlinelevel=level)
        h.addText(text)
        self.doc.text.addElement(h)
        self.toc_entries.append((level, text))

    def add_formatted_paragraph(self, parts):
        """
        Adiciona um parágrafo com partes formatadas
        parts: lista de tuplas (texto, style_name)
        """
        if not self.odf_available or not self.doc:
            return None

        p = P()

        for text, style_name in parts:
            if style_name:
                style = self.get_style_by_name(style_name)
                if style:
                    span = Span()
                    span.setAttribute('stylename', style)
                    span.addText(text)
                    p.addElement(span)
                else:
                    p.addText(text)
            else:
                p.addText(text)

        self.doc.text.addElement(p)
        return p

    def _get_style_name_for_value(self, value):
        """Retorna o nome do estilo apropriado para um determinado valor"""
        if isinstance(value, str):
            return "StringStyle"
        elif isinstance(value, (int, float)):
            return "NumberStyle"
        elif isinstance(value, bool):
            return "BooleanStyle"
        elif value is None:
            return "NullStyle"
        return None

    def format_json_to_paragraphs(self, data, indent=0):
        """
        Formata dados JSON em uma lista de parágrafos com formatação
        CORREÇÃO: Retorna uma lista de parágrafos, nunca elementos Span diretamente
        """
        paragraphs = []

        if isinstance(data, dict):
            if not data:
                # Dicionário vazio
                p = P()
                p.addText('{}')
                paragraphs.append(p)
            else:
                items = list(data.items())
                for i, (key, value) in enumerate(items):
                    # Cria um parágrafo para cada item
                    p = P()

                    # Adiciona indentação
                    if indent > 0:
                        p.addText('  ' * indent)

                    # Adiciona a chave formatada
                    key_span = Span()
                    key_style = self.get_style_by_name("KeyStyle")
                    if key_style:
                        key_span.setAttribute('stylename', key_style)
                    key_span.addText(f'"{key}"')
                    p.addElement(key_span)
                    p.addText(': ')

                    # Adiciona o valor
                    if isinstance(value, (dict, list)):
                        # Valor complexo - começa na mesma linha
                        if isinstance(value, dict):
                            p.addText('{')
                            paragraphs.append(p)
                            # Processa o conteúdo
                            sub_paragraphs = self.format_json_to_paragraphs(value, indent + 1)
                            paragraphs.extend(sub_paragraphs)
                            # Parágrafo de fechamento
                            closing_p = P()
                            closing_p.addText('  ' * indent + '}')
                            paragraphs.append(closing_p)
                        else:  # list
                            p.addText('[')
                            paragraphs.append(p)
                            sub_paragraphs = self.format_json_to_paragraphs(value, indent + 1)
                            paragraphs.extend(sub_paragraphs)
                            closing_p = P()
                            closing_p.addText('  ' * indent + ']')
                            paragraphs.append(closing_p)
                    else:
                        # Valor simples
                        style_name = self._get_style_name_for_value(value)
                        if style_name:
                            style = self.get_style_by_name(style_name)
                            if style:
                                span = Span()
                                span.setAttribute('stylename', style)
                                if isinstance(value, str):
                                    span.addText(f'"{value}"')
                                elif isinstance(value, bool):
                                    span.addText(str(value).lower())
                                elif value is None:
                                    span.addText('null')
                                else:
                                    span.addText(str(value))
                                p.addElement(span)
                            else:
                                if isinstance(value, str):
                                    p.addText(f'"{value}"')
                                else:
                                    p.addText(str(value))
                        else:
                            if isinstance(value, str):
                                p.addText(f'"{value}"')
                            else:
                                p.addText(str(value))
                        paragraphs.append(p)

                    # Adiciona vírgula se não for o último item
                    if i < len(items) - 1:
                        comma_p = P()
                        comma_p.addText('  ' * indent + ',')
                        paragraphs.append(comma_p)

        elif isinstance(data, list):
            if not data:
                # Lista vazia
                p = P()
                p.addText('[]')
                paragraphs.append(p)
            else:
                for i, item in enumerate(data):
                    p = P()
                    p.addText('  ' * indent + '- ')

                    if isinstance(item, (dict, list)):
                        p.addText('')
                        paragraphs.append(p)
                        sub_paragraphs = self.format_json_to_paragraphs(item, indent + 1)
                        paragraphs.extend(sub_paragraphs)
                    else:
                        style_name = self._get_style_name_for_value(item)
                        if style_name:
                            style = self.get_style_by_name(style_name)
                            if style:
                                span = Span()
                                span.setAttribute('stylename', style)
                                if isinstance(item, str):
                                    span.addText(f'"{item}"')
                                elif isinstance(item, bool):
                                    span.addText(str(item).lower())
                                elif item is None:
                                    span.addText('null')
                                else:
                                    span.addText(str(item))
                                p.addElement(span)
                            else:
                                if isinstance(item, str):
                                    p.addText(f'"{item}"')
                                else:
                                    p.addText(str(item))
                        else:
                            if isinstance(item, str):
                                p.addText(f'"{item}"')
                            else:
                                p.addText(str(item))
                        paragraphs.append(p)
        else:
            # Valor único
            p = P()
            style_name = self._get_style_name_for_value(data)
            if style_name:
                style = self.get_style_by_name(style_name)
                if style:
                    span = Span()
                    span.setAttribute('stylename', style)
                    if isinstance(data, str):
                        span.addText(f'"{data}"')
                    elif isinstance(data, bool):
                        span.addText(str(data).lower())
                    elif data is None:
                        span.addText('null')
                    else:
                        span.addText(str(data))
                    p.addElement(span)
                else:
                    if isinstance(data, str):
                        p.addText(f'"{data}"')
                    else:
                        p.addText(str(data))
            else:
                if isinstance(data, str):
                    p.addText(f'"{data}"')
                else:
                    p.addText(str(data))
            paragraphs.append(p)

        return paragraphs

    def json_to_odt(self, data, filename="output.odt"):
        """
        Converte dados JSON para documento ODT
        """
        # Se ODT não estiver disponível, salva como TXT formatado
        if not self.odf_available:
            print("⚠️ Gerando arquivo TXT formatado como alternativa...")
            txt_filename = filename.replace('.odt', '.txt')
            with open(txt_filename, 'w', encoding='utf-8') as f:
                f.write("Relatório de Dados JSON\n")
                f.write("=" * 50 + "\n")
                f.write(f"Data de exportação: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n")
                f.write(f"Tipo de dados: {type(data).__name__}\n")
                f.write("-" * 50 + "\n\n")
                f.write(json.dumps(data, indent=2, ensure_ascii=False))
            print(f"✅ Arquivo '{txt_filename}' gerado com sucesso!")
            return txt_filename

        # Cria documento
        self.create_document()

        # Adiciona cabeçalho com informações
        self.add_heading("Relatório de Dados JSON", level=1)

        # Adiciona metadados
        self.add_paragraph(f"Data de exportação: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
        self.add_paragraph(f"Tipo de dados: {type(data).__name__}")

        # Adiciona linha separadora
        self.add_paragraph("-" * 50)

        # Adiciona conteúdo JSON formatado
        self.add_heading("Conteúdo JSON", level=2)

        # Formata e adiciona o conteúdo como parágrafos
        paragraphs = self.format_json_to_paragraphs(data)
        for paragraph in paragraphs:
            self.doc.text.addElement(paragraph)

        # Salva o documento
        self.doc.save(filename)
        print(f"✅ Documento ODT salvo: {filename}")
        return filename

    def json_to_odt_table_format(self, data, filename="output_table.odt"):
        """
        Converte dados JSON para ODT usando formato de tabela
        """
        # Se ODT não estiver disponível, salva como CSV
        if not self.odf_available:
            print("⚠️ Gerando arquivo CSV como alternativa...")
            csv_filename = filename.replace('.odt', '.csv')

            if isinstance(data, list) and data and all(isinstance(item, dict) for item in data):
                import csv
                columns = list(data[0].keys())
                with open(csv_filename, 'w', encoding='utf-8', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=columns)
                    writer.writeheader()
                    writer.writerows(data)
                print(f"✅ Arquivo '{csv_filename}' gerado com sucesso!")
                return csv_filename
            else:
                txt_filename = filename.replace('.odt', '.txt')
                with open(txt_filename, 'w', encoding='utf-8') as f:
                    f.write(json.dumps(data, indent=2, ensure_ascii=False))
                print(f"✅ Arquivo '{txt_filename}' gerado com sucesso!")
                return txt_filename

        self.create_document()

        self.add_heading("Relatório de Dados JSON (Formato Tabela)", level=1)

        if isinstance(data, list) and data and all(isinstance(item, dict) for item in data):
            # Cria tabela para lista de dicionários
            self.add_heading("Dados em Formato de Tabela", level=2)

            # Determina colunas
            columns = list(data[0].keys())

            # Cria tabela
            table_element = table.Table()

            # Cabeçalho da tabela
            header_row = table.TableRow()
            for col in columns:
                header_cell = table.TableCell()
                header_p = P()
                header_p.addText(col)
                header_cell.addElement(header_p)
                header_row.addElement(header_cell)
            table_element.addElement(header_row)

            # Dados da tabela
            for item in data:
                row = table.TableRow()
                for col in columns:
                    cell = table.TableCell()
                    cell_p = P()

                    value = item.get(col, "")
                    if isinstance(value, (dict, list)):
                        cell_p.addText(json.dumps(value, ensure_ascii=False)[:100] + "...")
                    else:
                        cell_p.addText(str(value))

                    cell.addElement(cell_p)
                    row.addElement(cell)
                table_element.addElement(row)

            self.doc.text.addElement(table_element)

        else:
            # Para outros formatos, usa o formato padrão
            self.add_heading("Conteúdo JSON", level=2)
            paragraphs = self.format_json_to_paragraphs(data)
            for paragraph in paragraphs:
                self.doc.text.addElement(paragraph)

        self.doc.save(filename)
        print(f"✅ Documento ODT salvo: {filename}")
        return filename

def upload_json_file():
    """
    Faz upload de arquivo JSON do computador local para o Colab
    """
    print("📤 Selecione o(s) arquivo(s) JSON para upload:")
    uploaded = files.upload()

    files_list = []
    for filename in uploaded.keys():
        print(f"✅ Arquivo '{filename}' carregado com sucesso!")
        files_list.append(filename)

    return files_list

def read_json_file(filename):
    """
    Lê arquivo JSON e retorna os dados
    """
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            data = json.load(file)
        print(f"📖 Arquivo '{filename}' lido com sucesso!")
        return data
    except json.JSONDecodeError as e:
        print(f"❌ Erro ao decodificar JSON: {e}")
        return None
    except Exception as e:
        print(f"❌ Erro ao ler arquivo: {e}")
        return None

def download_file(filename):
    """
    Disponibiliza o arquivo para download
    """
    if os.path.exists(filename):
        print(f"📥 Preparando download do arquivo '{filename}'...")
        files.download(filename)
        print(f"✅ Download do arquivo '{filename}' iniciado!")
    else:
        print(f"❌ Arquivo '{filename}' não encontrado!")

def preview_json_content(data, max_lines=20):
    """
    Exibe uma prévia do conteúdo JSON
    """
    print("\n🔍 PRÉVIA DO CONTEÚDO JSON:")
    print("-" * 50)

    json_str = json.dumps(data, indent=2, ensure_ascii=False)
    lines = json_str.split('\n')

    for i, line in enumerate(lines[:max_lines]):
        if len(line) > 100:
            line = line[:97] + "..."
        print(line)

    if len(lines) > max_lines:
        print(f"... e mais {len(lines) - max_lines} linhas")

    print("-" * 50)

def main():
    """
    Função principal do script
    """
    print("🚀 Conversor JSON para ODT - Google Colab")
    print("=" * 50)

    # 1. Upload do(s) arquivo(s) JSON
    json_files = upload_json_file()

    if not json_files:
        print("❌ Nenhum arquivo foi carregado.")
        return

    # 2. Menu de opções para cada arquivo
    for json_file in json_files:
        print(f"\n📄 Processando arquivo: {json_file}")
        print("-" * 50)

        # Lê o arquivo JSON
        data = read_json_file(json_file)
        if data is None:
            continue

        # Exibe prévia do conteúdo
        preview_json_content(data)

        # Opções de conversão
        print("\n📋 OPÇÕES DE CONVERSÃO:")
        print("1. Formato hierárquico com indentação (estilo código)")
        print("2. Formato tabela (recomendado para listas de objetos)")
        print("3. Ambos os formatos")

        option = input("\n👉 Escolha uma opção (1-3): ").strip()

        # Nome base para os arquivos de saída
        base_name = os.path.splitext(json_file)[0]

        converter = JSONtoODTConverter()

        if option == '1' or option == '3':
            output_file = f"{base_name}_hierarquico.odt"
            print(f"\n📝 Convertendo para formato hierárquico...")
            result = converter.json_to_odt(data, output_file)
            print(f"✅ Arquivo '{result}' gerado com sucesso!")
            download_file(result)

        if option == '2' or option == '3':
            output_file = f"{base_name}_tabela.odt"
            print(f"\n📝 Convertendo para formato tabela...")
            result = converter.json_to_odt_table_format(data, output_file)
            print(f"✅ Arquivo '{result}' gerado com sucesso!")
            download_file(result)

        if option not in ['1', '2', '3']:
            print("❌ Opção inválida!")
            continue

        print(f"\n✅ Processamento do arquivo '{json_file}' concluído!")

    print("\n🎉 Todos os arquivos foram processados com sucesso!")

def batch_convert_to_odt():
    """
    Função para conversão em lote de múltiplos arquivos JSON
    """
    print("🚀 Conversão em Lote JSON para ODT")
    print("=" * 50)

    json_files = upload_json_file()

    if not json_files:
        print("❌ Nenhum arquivo foi carregado.")
        return

    format_type = input("Escolha o formato (1=hierárquico, 2=tabela, 3=ambos): ")

    converter = JSONtoODTConverter()

    for json_file in json_files:
        print(f"\n📄 Processando: {json_file}")

        data = read_json_file(json_file)
        if data is None:
            continue

        base_name = os.path.splitext(json_file)[0]

        if format_type == '1' or format_type == '3':
            output_file = f"{base_name}_hierarquico.odt"
            result = converter.json_to_odt(data, output_file)
            download_file(result)

        if format_type == '2' or format_type == '3':
            output_file = f"{base_name}_tabela.odt"
            result = converter.json_to_odt_table_format(data, output_file)
            download_file(result)

    print("\n✅ Conversão em lote concluída!")

# Executar o script principal
if __name__ == "__main__":
    print("=" * 50)
    print("Selecione o modo de execução:")
    print("1. Modo normal (processar um arquivo por vez)")
    print("2. Modo em lote (processar múltiplos arquivos)")

    mode = input("\n👉 Escolha o modo (1 ou 2): ").strip()

    if mode == '2':
        batch_convert_to_odt()
    else:
        main()

✅ Biblioteca odfpy já está instalada!
Selecione o modo de execução:
1. Modo normal (processar um arquivo por vez)
2. Modo em lote (processar múltiplos arquivos)

👉 Escolha o modo (1 ou 2): 1
🚀 Conversor JSON para ODT - Google Colab
📤 Selecione o(s) arquivo(s) JSON para upload:


Saving conversations.json to conversations (2).json
✅ Arquivo 'conversations (2).json' carregado com sucesso!

📄 Processando arquivo: conversations (2).json
--------------------------------------------------
📖 Arquivo 'conversations (2).json' lido com sucesso!

🔍 PRÉVIA DO CONTEÚDO JSON:
--------------------------------------------------
[
  {
    "id": "6a4a5f5a-725d-413d-935f-939017d4613c",
    "title": "Fisiologia da ejaculação masculina",
    "inserted_at": "2026-03-29T23:13:19.712000+08:00",
    "updated_at": "2026-03-30T00:00:49.973000+08:00",
    "mapping": {
      "root": {
        "id": "root",
        "parent": null,
        "children": [
          "1"
        ],
        "message": null
      },
      "1": {
        "id": "1",
        "parent": "root",
        "children": [
          "2"
... e mais 524 linhas
--------------------------------------------------

📋 OPÇÕES DE CONVERSÃO:
1. Formato hierárquico com indentação (estilo código)
2. Formato tabela (recomendado para list

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download do arquivo 'conversations (2)_hierarquico.odt' iniciado!

✅ Processamento do arquivo 'conversations (2).json' concluído!

🎉 Todos os arquivos foram processados com sucesso!
